# vecsearch GPU benchmark (Colab)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 15 minutes).
Builds vecsearch with CUDA, runs the GPU tests, then times exact k-NN on 1M x 384 vectors with the CPU index, the CUDA index and FAISS. Downloads `gpu_results.zip` at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep -E 'Model name|^CPU\(s\)|avx2' | cut -c1-120
!free -g | head -2

In [ ]:
BRANCH = 'main'
!rm -rf vecsearch && git clone -q -b {BRANCH} https://github.com/Manojkaipu/vecsearch
%cd vecsearch
!git log --oneline -1
!pip install -q faiss-gpu-cu12 matplotlib pytest
!CMAKE_ARGS='-DVECSEARCH_BUILD_CUDA=ON' pip install -q . 2>&1 | tail -5
import vecsearch as vs; print('cuda available:', vs.cuda_available())

In [ ]:
!python -m pytest tests/python -q 2>&1 | tail -5

Kernel ablation: the same search with each distance kernel.

In [ ]:
!python bench/gpu_bench.py --backends cuda-naive,cuda-skinny,cuda-tiled --batches 1,4,8,16,32,128,1024 --max-batch cuda-skinny=128 --check 64 --out results/gpu/colab_methods.csv

Main comparison. The CPU here is Colab's 2 vCPUs; the README also reports the laptop CPU.

In [ ]:
!python bench/gpu_bench.py --backends cuda,faiss-gpu,cpu,faiss-cpu --out results/gpu/colab.csv

In [ ]:
!python bench/plot_gpu.py results/gpu/colab.csv --out results/gpu/colab_qps.png
!python bench/plot_gpu.py results/gpu/colab_methods.csv --out results/gpu/colab_methods.png
!zip -qr /content/gpu_results.zip results/gpu
from google.colab import files; files.download('/content/gpu_results.zip')